In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
from typing import Iterable
import warnings

import nibabel as nib
import numpy as np
import pandas as pd


def _load_nifti(path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load a NIfTI while preserving complex-valued data, if present."""
    image = nib.load(str(path))
    data = np.asanyarray(image.dataobj)
    return np.asarray(data), image.affine


def _check_affine(
    affine: np.ndarray,
    reference_affine: np.ndarray,
    path: Path,
    atol: float = 1e-3,
) -> None:
    if not np.allclose(affine, reference_affine, atol=atol):
        raise ValueError(
            f"Affine mismatch for:\n{path}\n"
            "The image is apparently not on the same grid as the brain mask."
        )


def _prepare_spatial_map(
    data: np.ndarray,
    expected_shape: tuple[int, ...],
    path: Path,
) -> np.ndarray:
    """Remove singleton dimensions and verify the spatial shape."""
    data = np.squeeze(data)

    if data.shape != expected_shape:
        raise ValueError(
            f"Unexpected shape for {path}:\n"
            f"  found:   {data.shape}\n"
            f"  expected: {expected_shape}"
        )

    return data


def _find_spectral_axis(
    spectrum_shape: tuple[int, ...],
    spatial_shape: tuple[int, ...],
) -> int:
    """
    Find the spectral axis by checking which axis can be removed to obtain
    the brain-mask shape.
    """
    candidates = []

    for axis in range(len(spectrum_shape)):
        reduced_shape = (
            spectrum_shape[:axis]
            + spectrum_shape[axis + 1:]
        )

        if reduced_shape == spatial_shape:
            candidates.append(axis)

    if not candidates:
        raise ValueError(
            "Could not identify the spectral axis.\n"
            f"Spectrum shape: {spectrum_shape}\n"
            f"Mask shape:     {spatial_shape}"
        )

    # Prefer the last axis if several dimensions happen to have equal lengths.
    if len(spectrum_shape) - 1 in candidates:
        return len(spectrum_shape) - 1

    if len(candidates) > 1:
        raise ValueError(
            "The spectral axis is ambiguous.\n"
            f"Spectrum shape: {spectrum_shape}\n"
            f"Mask shape:     {spatial_shape}\n"
            f"Candidate axes: {candidates}\n"
            "Pass spectra with unambiguous dimensions or adapt the helper."
        )

    return candidates[0]


def collect_relative_metabolite_amplitudes(
    subject_paths: Iterable[str | Path],
    *,
    crlb_threshold: float = 20.0,
    mask_filename: str = "mask.nii.gz",
    fit_filename: str = "SpecMap_LCMFit.nii.gz",
    baseline_filename: str = "SpecMap_LCMBaseline.nii.gz",
    maps_subdirectory: str = "Orig",
    amp_suffix: str = "_amp_map.nii.gz",
    sd_suffix: str = "_sd_map.nii.gz",
    mask_threshold: float = 0.5,
    denominator_epsilon: float = 1e-12,
    subtract_baseline: bool = True,
    check_affines: bool = True,
    strict: bool = True,
) -> tuple[dict[str, np.ndarray], pd.DataFrame]:
    """
    Collect relative LCModel amplitudes from several subjects.

    For each valid voxel and metabolite:

        relative_amplitude =
            amplitude_map / max(abs(metabolite_fit_spectrum))

    where, by default:

        metabolite_fit_spectrum =
            SpecMap_LCMFit - SpecMap_LCMBaseline

    Voxels are retained when:
      - they are inside the brain mask,
      - the CRLB is finite and <= crlb_threshold,
      - the amplitude is finite,
      - the spectral normalization factor is finite and nonzero.

    Parameters
    ----------
    subject_paths
        Directories containing mask.nii.gz, SpecMap_LCMFit.nii.gz,
        SpecMap_LCMBaseline.nii.gz and the Orig directory.
    crlb_threshold
        Maximum accepted value in the *_sd_map.nii.gz files.
    strict
        If True, missing CRLB maps or missing required files raise errors.
        If False, affected metabolites or subjects are skipped with warnings.

    Returns
    -------
    relative_amplitudes
        Dictionary mapping metabolite names to pooled one-dimensional arrays.
    qc_table
        Per-subject and per-metabolite summary of accepted voxels.
    """
    collected: dict[str, list[np.ndarray]] = defaultdict(list)
    qc_rows: list[dict] = []

    subject_paths = [Path(path) for path in subject_paths]

    if not subject_paths:
        raise ValueError("subject_paths is empty.")

    for subject_path in subject_paths:
        mask_path = subject_path / mask_filename
        fit_path = subject_path / fit_filename
        baseline_path = subject_path / baseline_filename
        maps_path = subject_path / maps_subdirectory

        required_paths = [mask_path, fit_path, maps_path]

        if subtract_baseline:
            required_paths.append(baseline_path)

        missing_paths = [
            path for path in required_paths
            if not path.exists()
        ]

        if missing_paths:
            message = (
                f"Missing required files/directories for "
                f"{subject_path.name}:\n"
                + "\n".join(str(path) for path in missing_paths)
            )

            if strict:
                raise FileNotFoundError(message)

            warnings.warn(message)
            continue

        # ------------------------------------------------------------------
        # Brain mask
        # ------------------------------------------------------------------
        mask_data, mask_affine = _load_nifti(mask_path)
        mask_data = np.squeeze(mask_data)

        if mask_data.ndim != 3:
            raise ValueError(
                f"Expected a 3D brain mask, found {mask_data.shape} "
                f"for {mask_path}."
            )

        brain_mask = (
            np.isfinite(mask_data)
            & (mask_data > mask_threshold)
        )

        spatial_shape = brain_mask.shape

        # ------------------------------------------------------------------
        # Fitted spectrum and metabolite-only spectral normalization
        # ------------------------------------------------------------------
        fit_data, fit_affine = _load_nifti(fit_path)
        fit_data = np.squeeze(fit_data)

        if check_affines:
            _check_affine(fit_affine, mask_affine, fit_path)

        if subtract_baseline:
            baseline_data, baseline_affine = _load_nifti(
                baseline_path
            )
            baseline_data = np.squeeze(baseline_data)

            if check_affines:
                _check_affine(
                    baseline_affine,
                    mask_affine,
                    baseline_path,
                )

            if fit_data.shape != baseline_data.shape:
                raise ValueError(
                    "Fit and baseline spectra have different shapes:\n"
                    f"  fit:      {fit_data.shape}\n"
                    f"  baseline: {baseline_data.shape}"
                )

            normalization_spectra = fit_data - baseline_data
        else:
            normalization_spectra = fit_data

        spectral_axis = _find_spectral_axis(
            normalization_spectra.shape,
            spatial_shape,
        )

        max_abs_spectrum = np.max(
            np.abs(normalization_spectra),
            axis=spectral_axis,
        )

        if max_abs_spectrum.shape != spatial_shape:
            raise RuntimeError(
                "Internal shape error after reducing the spectral axis:\n"
                f"  result: {max_abs_spectrum.shape}\n"
                f"  mask:   {spatial_shape}"
            )

        # ------------------------------------------------------------------
        # Metabolite amplitude and CRLB maps
        # ------------------------------------------------------------------
        amplitude_paths = sorted(
            maps_path.glob(f"*{amp_suffix}")
        )

        if not amplitude_paths:
            message = (
                f"No files ending in '{amp_suffix}' found in "
                f"{maps_path}."
            )

            if strict:
                raise FileNotFoundError(message)

            warnings.warn(message)
            continue

        for amplitude_path in amplitude_paths:
            metabolite = amplitude_path.name[:-len(amp_suffix)]

            crlb_path = amplitude_path.with_name(
                f"{metabolite}{sd_suffix}"
            )

            if not crlb_path.exists():
                message = (
                    f"Missing CRLB map for metabolite "
                    f"'{metabolite}': {crlb_path}"
                )

                if strict:
                    raise FileNotFoundError(message)

                warnings.warn(message)
                continue

            amplitude_data, amplitude_affine = _load_nifti(
                amplitude_path
            )
            crlb_data, crlb_affine = _load_nifti(crlb_path)

            if check_affines:
                _check_affine(
                    amplitude_affine,
                    mask_affine,
                    amplitude_path,
                )
                _check_affine(
                    crlb_affine,
                    mask_affine,
                    crlb_path,
                )

            amplitude_data = _prepare_spatial_map(
                amplitude_data,
                spatial_shape,
                amplitude_path,
            )

            crlb_data = _prepare_spatial_map(
                crlb_data,
                spatial_shape,
                crlb_path,
            )

            crlb_valid = (
                np.isfinite(crlb_data)
                & (crlb_data >= 0)
                & (crlb_data <= crlb_threshold)
            )

            denominator_valid = (
                np.isfinite(max_abs_spectrum)
                & (max_abs_spectrum > denominator_epsilon)
            )

            valid = (
                brain_mask
                & crlb_valid
                & denominator_valid
                & np.isfinite(amplitude_data)
            )

            relative_values = (
                amplitude_data[valid]
                / max_abs_spectrum[valid]
            ).astype(np.float64, copy=False)

            collected[metabolite].append(relative_values)

            qc_rows.append(
                {
                    "subject": subject_path.name,
                    "metabolite": metabolite,
                    "n_brain_voxels": int(brain_mask.sum()),
                    "n_crlb_pass": int(
                        (brain_mask & crlb_valid).sum()
                    ),
                    "n_valid": int(valid.sum()),
                    "median_relative_amplitude": (
                        float(np.median(relative_values))
                        if relative_values.size
                        else np.nan
                    ),
                    "q1_relative_amplitude": (
                        float(np.quantile(relative_values, 0.25))
                        if relative_values.size
                        else np.nan
                    ),
                    "q3_relative_amplitude": (
                        float(np.quantile(relative_values, 0.75))
                        if relative_values.size
                        else np.nan
                    ),
                }
            )

    relative_amplitudes = {
        metabolite: np.concatenate(arrays)
        for metabolite, arrays in sorted(collected.items())
        if arrays
    }

    qc_table = pd.DataFrame(qc_rows)

    return relative_amplitudes, qc_table

In [ ]:
healthy_subjects = [
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Vienna/Vol09_Dat_NoL2_GradDel/maps",
    #"/path/to/Healthy02",
    #"/path/to/Healthy03",
]

healthy_amplitudes, healthy_qc = (
    collect_relative_metabolite_amplitudes(
        healthy_subjects,
        crlb_threshold=30.0,
    )
)

In [ ]:
healthy_amplitudes.keys()

In [ ]:
naa = healthy_amplitudes["NAA"]

print("Anzahl:", naa.size)
print("Median:", np.median(naa))
print("IQR:", np.quantile(naa, 0.75) - np.quantile(naa, 0.25))

In [ ]:
import matplotlib.pyplot as plt

metabolite = "NAA"
values = healthy_amplitudes[metabolite]

plt.figure(figsize=(7, 4))
plt.hist(values, bins=100, density=True)
plt.axvline(
    np.median(values),
    linestyle="--",
    label=f"Median = {np.median(values):.3g}",
)
plt.xlabel(
    "Amplitude / max abs metabolite-fit spectrum"
)
plt.ylabel("Density")
plt.title(metabolite)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from __future__ import annotations

import math
from typing import Mapping

import matplotlib.pyplot as plt
import numpy as np


def plot_all_metabolite_histograms(
    amplitudes: Mapping[str, np.ndarray],
    *,
    n_columns: int = 4,
    bins: int = 60,
    density: bool = True,
    title: str | None = "Relative metabolite amplitudes",
    statistics_threshold: float | None = 0.0,
    display_quantiles: tuple[float, float] | None = None,
    figsize_per_panel: tuple[float, float] = (4.0, 3.0),
) -> tuple[plt.Figure, np.ndarray]:
    """
    Plot one histogram per metabolite in a raster.

    Parameters
    ----------
    amplitudes
        Dictionary of the form:

            {
                "NAA": np.ndarray(...),
                "Cr": np.ndarray(...),
                ...
            }

    n_columns
        Number of subplot columns.

    bins
        Number of histogram bins.

    density
        If True, plot probability density rather than voxel counts.

    title
        Figure title.

    statistics_threshold
        Median and IQR are calculated only from values strictly greater than
        this threshold.

        Examples
        --------
        0.0:
            Ignore zeros and negative values for Median/IQR.

        None:
            Use all finite values for Median/IQR.

    display_quantiles
        Optional quantile limits for the displayed x-axis, for example:

            (0.005, 0.995)

        This only changes the displayed x-axis. Values are not removed from
        the returned arrays.

        Use None to display the complete range.

    figsize_per_panel
        Width and height per subplot in inches.

    Returns
    -------
    fig
        Matplotlib figure.

    axes
        Two-dimensional array of subplot axes.
    """
    metabolite_names = sorted(amplitudes)

    if not metabolite_names:
        raise ValueError("The amplitudes dictionary is empty.")

    if n_columns < 1:
        raise ValueError("n_columns must be at least 1.")

    n_metabolites = len(metabolite_names)
    n_rows = math.ceil(n_metabolites / n_columns)

    fig, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(
            figsize_per_panel[0] * n_columns,
            figsize_per_panel[1] * n_rows,
        ),
        squeeze=False,
        constrained_layout=True,
    )

    flat_axes = axes.ravel()

    for axis, metabolite in zip(flat_axes, metabolite_names):
        values = np.asarray(
            amplitudes[metabolite],
            dtype=np.float64,
        ).ravel()

        values = values[np.isfinite(values)]

        if values.size == 0:
            axis.set_title(metabolite)
            axis.text(
                0.5,
                0.5,
                "No valid values",
                horizontalalignment="center",
                verticalalignment="center",
                transform=axis.transAxes,
            )
            axis.set_axis_off()
            continue

        # Values used for robust summary statistics.
        if statistics_threshold is None:
            statistics_values = values
        else:
            statistics_values = values[
                values > statistics_threshold
            ]

        axis.hist(
            values,
            bins=bins,
            density=density,
            alpha=0.75,
        )

        # Optional display range, without changing the actual data.
        if display_quantiles is not None:
            lower_quantile, upper_quantile = display_quantiles

            if not (
                0 <= lower_quantile
                < upper_quantile
                <= 1
            ):
                raise ValueError(
                    "display_quantiles must satisfy "
                    "0 <= lower < upper <= 1."
                )

            x_min, x_max = np.quantile(
                values,
                [lower_quantile, upper_quantile],
            )

            if np.isfinite(x_min) and np.isfinite(x_max):
                if x_max > x_min:
                    axis.set_xlim(x_min, x_max)

        if statistics_values.size > 0:
            q1, median, q3 = np.quantile(
                statistics_values,
                [0.25, 0.50, 0.75],
            )

            axis.axvline(
                median,
                linestyle="-",
                linewidth=1.5,
                label=f"Median: {median:.3g}",
            )

            axis.axvline(
                q1,
                linestyle="--",
                linewidth=1.0,
            )

            axis.axvline(
                q3,
                linestyle="--",
                linewidth=1.0,
                label=f"IQR: {q3 - q1:.3g}",
            )

        zero_fraction = np.mean(values <= 0)

        axis.set_title(
            f"{metabolite}\n"
            f"n={values.size:,}, ≤0={zero_fraction:.1%}"
        )

        axis.set_xlabel(
            "Relative amplitude"
        )

        axis.set_ylabel(
            "Density" if density else "Voxel count"
        )

        axis.legend(
            fontsize="small",
            frameon=False,
        )

        axis.grid(
            axis="y",
            alpha=0.2,
        )

    # Hide unused axes in the final row.
    for axis in flat_axes[n_metabolites:]:
        axis.set_visible(False)

    if title is not None:
        fig.suptitle(
            title,
            fontsize=15,
        )

    return fig, axes

In [ ]:
fig, axes = plot_all_metabolite_histograms(
    healthy_amplitudes,
    n_columns=2,
    bins=60,
    title="Healthy – relative metabolite amplitudes",
    statistics_threshold=0.0,
)

plt.show()

In [ ]:
from __future__ import annotations

import math
from typing import Mapping

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import truncnorm


def plot_all_metabolite_histograms(
    amplitudes: Mapping[str, np.ndarray],
    *,
    n_columns: int = 4,
    bins: int = 60,
    title: str | None = "Relative metabolite amplitudes",
    width_factor: float = 1.0,
    statistics_threshold: float = 0.0,
    display_quantiles: tuple[float, float] | None = (0.001, 0.999),
    figsize_per_panel: tuple[float, float] = (4.2, 3.2),
) -> tuple[plt.Figure, np.ndarray]:
    """
    Plot all metabolite amplitude histograms with an overlaid
    zero-truncated normal simulation distribution.

    Simulation parameters
    ---------------------
    mean:
        Median of all finite observed amplitudes.

    standard deviation:
        width_factor * IQR of amplitudes above statistics_threshold.

    The normal distribution is truncated at zero. This is mathematically
    equivalent to repeatedly sampling from N(mean, std²) until a
    non-negative value is obtained.
    """
    metabolite_names = sorted(amplitudes)

    if not metabolite_names:
        raise ValueError("The amplitudes dictionary is empty.")

    if n_columns < 1:
        raise ValueError("n_columns must be at least 1.")

    if width_factor <= 0:
        raise ValueError("width_factor must be positive.")

    n_metabolites = len(metabolite_names)
    n_rows = math.ceil(n_metabolites / n_columns)

    fig, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(
            figsize_per_panel[0] * n_columns,
            figsize_per_panel[1] * n_rows,
        ),
        squeeze=False,
        constrained_layout=True,
    )

    flat_axes = axes.ravel()

    for axis, metabolite in zip(flat_axes, metabolite_names):
        values = np.asarray(
            amplitudes[metabolite],
            dtype=np.float64,
        ).ravel()

        values = values[np.isfinite(values)]

        if values.size == 0:
            axis.set_title(metabolite)
            axis.text(
                0.5,
                0.5,
                "No valid values",
                ha="center",
                va="center",
                transform=axis.transAxes,
            )
            axis.set_axis_off()
            continue

        # Centre from all observed values.
        median = float(np.median(values))

        # Spread from positive / reliably present values only.
        spread_values = values[
            values > statistics_threshold
        ]

        if spread_values.size >= 4:
            q1, q3 = np.quantile(
                spread_values,
                [0.25, 0.75],
            )
            iqr = float(q3 - q1)
        else:
            q1 = q3 = iqr = np.nan

        simulation_std = width_factor * iqr

        # Determine a useful visible x-range.
        if display_quantiles is None:
            x_min = float(np.min(values))
            x_max = float(np.max(values))
        else:
            lower_q, upper_q = display_quantiles

            if not 0 <= lower_q < upper_q <= 1:
                raise ValueError(
                    "display_quantiles must satisfy "
                    "0 <= lower < upper <= 1."
                )

            x_min, x_max = np.quantile(
                values,
                [lower_q, upper_q],
            )

        # Include enough of the deliberately broadened simulation distribution.
        if np.isfinite(simulation_std) and simulation_std > 0:
            x_max = max(
                x_max,
                median + 4.0 * simulation_std,
            )

        x_min = max(0.0, float(x_min))
        x_max = float(x_max)

        axis.hist(
            values,
            bins=bins,
            range=(x_min, x_max),
            density=True,
            alpha=0.65,
            label="In vivo",
        )

        # Overlay the actual zero-truncated simulation density.
        if np.isfinite(simulation_std) and simulation_std > 0:
            lower_standardized = (
                0.0 - median
            ) / simulation_std

            simulation_distribution = truncnorm(
                a=lower_standardized,
                b=np.inf,
                loc=median,
                scale=simulation_std,
            )

            x = np.linspace(
                0.0,
                x_max,
                500,
            )

            axis.plot(
                x,
                simulation_distribution.pdf(x),
                linewidth=2.0,
                label=(
                    "Simulation\n"
                    f"μ={median:.3g}, "
                    f"σ={simulation_std:.3g}"
                ),
            )

        axis.axvline(
            median,
            linestyle="--",
            linewidth=1.2,
            label=f"In-vivo median: {median:.3g}",
        )

        axis.set_xlim(x_min, x_max)

        axis.set_title(
            f"{metabolite}\n"
            f"n={values.size:,}, "
            f"IQR={iqr:.3g}"
        )

        axis.set_xlabel("Relative amplitude")
        axis.set_ylabel("Density")
        axis.grid(axis="y", alpha=0.2)
        axis.legend(fontsize="small", frameon=False)

    for axis in flat_axes[n_metabolites:]:
        axis.set_visible(False)

    if title is not None:
        fig.suptitle(title, fontsize=15)

    return fig, axes

In [ ]:
fig, axes = plot_all_metabolite_histograms(
    healthy_amplitudes,
    n_columns=4,
    bins=60,
    width_factor=2,
    title=(
        "Healthy amplitudes and proposed "
        "simulation distributions"
    ),
)

plt.show()

In [ ]:
from __future__ import annotations

from collections.abc import Iterable
from pathlib import Path
import warnings

import nibabel as nib
import numpy as np
import pandas as pd


def _load_spatial_nifti(
    path: Path,
    *,
    expected_shape: tuple[int, ...] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Load a real-valued spatial NIfTI map and remove singleton dimensions.
    """
    image = nib.load(str(path))
    data = np.asarray(image.dataobj)
    data = np.squeeze(data)

    if data.ndim != 3:
        raise ValueError(
            f"Expected a 3D spatial map:\n"
            f"  path:  {path}\n"
            f"  shape: {data.shape}"
        )

    if expected_shape is not None and data.shape != expected_shape:
        raise ValueError(
            f"Shape mismatch:\n"
            f"  path:     {path}\n"
            f"  found:    {data.shape}\n"
            f"  expected: {expected_shape}"
        )

    return data, image.affine


def _check_affine(
    affine: np.ndarray,
    reference_affine: np.ndarray,
    path: Path,
    *,
    atol: float = 1e-3,
) -> None:
    if not np.allclose(
        affine,
        reference_affine,
        atol=atol,
    ):
        raise ValueError(
            f"Affine mismatch for:\n{path}"
        )


def _build_crlb_mask(
    *,
    orig_directory: Path,
    metabolites: tuple[str, ...] | None,
    threshold: float,
    spatial_shape: tuple[int, ...],
    reference_affine: np.ndarray,
    mode: str,
    sd_suffix: str,
    check_affines: bool,
    strict: bool,
) -> np.ndarray:
    """
    Build a joint voxel mask from one or more CRLB maps.

    mode="any":
        At least one selected metabolite must pass the threshold.

    mode="all":
        Every selected metabolite must pass the threshold.

    metabolites=None:
        No CRLB filtering is applied.
    """
    if metabolites is None:
        return np.ones(
            spatial_shape,
            dtype=bool,
        )

    if mode not in {"any", "all"}:
        raise ValueError(
            "mode must be either 'any' or 'all'."
        )

    metabolite_masks = []

    for metabolite in metabolites:
        crlb_path = (
            orig_directory
            / f"{metabolite}{sd_suffix}"
        )

        if not crlb_path.exists():
            message = (
                f"Missing CRLB map for "
                f"'{metabolite}':\n{crlb_path}"
            )

            if strict:
                raise FileNotFoundError(message)

            warnings.warn(message)
            continue

        crlb, crlb_affine = _load_spatial_nifti(
            crlb_path,
            expected_shape=spatial_shape,
        )

        if check_affines:
            _check_affine(
                crlb_affine,
                reference_affine,
                crlb_path,
            )

        valid = (
            np.isfinite(crlb)
            & (crlb >= 0)
            & (crlb <= threshold)
        )

        metabolite_masks.append(valid)

    if not metabolite_masks:
        raise ValueError(
            "No usable CRLB maps were found."
        )

    stacked_masks = np.stack(
        metabolite_masks,
        axis=0,
    )

    if mode == "any":
        return np.any(
            stacked_masks,
            axis=0,
        )

    return np.all(
        stacked_masks,
        axis=0,
    )


def collect_lcmodel_fit_parameters(
    subject_paths: Iterable[str | Path],
    *,
    crlb_threshold: float = 20.0,
    crlb_metabolites: tuple[str, ...] | None = (
        "NAA",
    ),
    crlb_mode: str = "any",
    mask_filename: str = "mask.nii.gz",
    extra_subdirectory: str = "Extra",
    orig_subdirectory: str = "Orig",
    fwhm_filename: str = "FWHM_map.nii.gz",
    shift_filename: str = "shift_map.nii.gz",
    sd_suffix: str = "_sd_map.nii.gz",
    mask_threshold: float = 0.5,
    require_positive_fwhm: bool = True,
    check_affines: bool = True,
    strict: bool = True,
) -> tuple[dict[str, np.ndarray], pd.DataFrame]:
    """
    Collect voxelwise LCModel FWHM and frequency-shift values.

    Voxels must:
      - be inside the brain mask,
      - pass the selected CRLB criterion,
      - contain finite parameter values,
      - have positive FWHM when require_positive_fwhm=True.

    Returns
    -------
    parameters
        {
            "FWHM": pooled_fwhm_values,
            "shift": pooled_shift_values,
        }

    qc_table
        Per-subject voxel counts and robust parameter summaries.
    """
    subject_paths = [
        Path(path)
        for path in subject_paths
    ]

    if not subject_paths:
        raise ValueError(
            "subject_paths is empty."
        )

    collected_fwhm = []
    collected_shift = []
    qc_rows = []

    for subject_path in subject_paths:
        mask_path = (
            subject_path
            / mask_filename
        )

        extra_directory = (
            subject_path
            / extra_subdirectory
        )

        orig_directory = (
            subject_path
            / orig_subdirectory
        )

        fwhm_path = (
            extra_directory
            / fwhm_filename
        )

        shift_path = (
            extra_directory
            / shift_filename
        )

        required_paths = [
            mask_path,
            extra_directory,
            orig_directory,
            fwhm_path,
            shift_path,
        ]

        missing = [
            path
            for path in required_paths
            if not path.exists()
        ]

        if missing:
            message = (
                f"Missing files/directories for "
                f"{subject_path.name}:\n"
                + "\n".join(
                    str(path)
                    for path in missing
                )
            )

            if strict:
                raise FileNotFoundError(message)

            warnings.warn(message)
            continue

        # --------------------------------------------------------------
        # Brain mask
        # --------------------------------------------------------------
        mask_data, mask_affine = (
            _load_spatial_nifti(mask_path)
        )

        brain_mask = (
            np.isfinite(mask_data)
            & (mask_data > mask_threshold)
        )

        spatial_shape = brain_mask.shape

        # --------------------------------------------------------------
        # CRLB quality mask
        # --------------------------------------------------------------
        crlb_mask = _build_crlb_mask(
            orig_directory=orig_directory,
            metabolites=crlb_metabolites,
            threshold=crlb_threshold,
            spatial_shape=spatial_shape,
            reference_affine=mask_affine,
            mode=crlb_mode,
            sd_suffix=sd_suffix,
            check_affines=check_affines,
            strict=strict,
        )

        common_valid = (
            brain_mask
            & crlb_mask
        )

        # --------------------------------------------------------------
        # FWHM
        # --------------------------------------------------------------
        fwhm, fwhm_affine = (
            _load_spatial_nifti(
                fwhm_path,
                expected_shape=spatial_shape,
            )
        )

        if check_affines:
            _check_affine(
                fwhm_affine,
                mask_affine,
                fwhm_path,
            )

        fwhm_valid = (
            common_valid
            & np.isfinite(fwhm)
        )

        if require_positive_fwhm:
            fwhm_valid &= fwhm > 0

        fwhm_values = fwhm[
            fwhm_valid
        ].astype(
            np.float64,
            copy=False,
        )

        # --------------------------------------------------------------
        # Frequency shift
        # --------------------------------------------------------------
        shift, shift_affine = (
            _load_spatial_nifti(
                shift_path,
                expected_shape=spatial_shape,
            )
        )

        if check_affines:
            _check_affine(
                shift_affine,
                mask_affine,
                shift_path,
            )

        shift_valid = (
            common_valid
            & np.isfinite(shift)
        )

        shift_values = shift[
            shift_valid
        ].astype(
            np.float64,
            copy=False,
        )

        collected_fwhm.append(
            fwhm_values
        )

        collected_shift.append(
            shift_values
        )

        qc_rows.append(
            {
                "subject": subject_path.name,
                "n_brain_voxels": int(
                    brain_mask.sum()
                ),
                "n_crlb_pass": int(
                    common_valid.sum()
                ),
                "n_fwhm_valid": int(
                    fwhm_values.size
                ),
                "n_shift_valid": int(
                    shift_values.size
                ),
                "median_fwhm": (
                    float(
                        np.median(fwhm_values)
                    )
                    if fwhm_values.size
                    else np.nan
                ),
                "iqr_fwhm": (
                    float(
                        np.quantile(
                            fwhm_values,
                            0.75,
                        )
                        - np.quantile(
                            fwhm_values,
                            0.25,
                        )
                    )
                    if fwhm_values.size
                    else np.nan
                ),
                "median_shift": (
                    float(
                        np.median(shift_values)
                    )
                    if shift_values.size
                    else np.nan
                ),
                "iqr_shift": (
                    float(
                        np.quantile(
                            shift_values,
                            0.75,
                        )
                        - np.quantile(
                            shift_values,
                            0.25,
                        )
                    )
                    if shift_values.size
                    else np.nan
                ),
            }
        )

    parameters = {
        "FWHM": (
            np.concatenate(collected_fwhm)
            if collected_fwhm
            else np.empty(
                0,
                dtype=np.float64,
            )
        ),
        "shift": (
            np.concatenate(collected_shift)
            if collected_shift
            else np.empty(
                0,
                dtype=np.float64,
            )
        ),
    }

    qc_table = pd.DataFrame(
        qc_rows
    )

    return parameters, qc_table

In [ ]:
healthy_subjects = [
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/home/bstrasser/Projects/Project9_ImplementRecoInICE/Step5_MultiCenterStudy/LargeData_d3hj/Results/Vienna/Vol09_Dat_NoL2_GradDel/maps",
   # "/path/to/Healthy02",
    #"/path/to/Healthy03",
]

healthy_fit_parameters, healthy_fit_qc = (
    collect_lcmodel_fit_parameters(
        healthy_subjects,
        crlb_threshold=20.0,
        crlb_metabolites=(
            "NAA+NAAG",
        ),
        crlb_mode="any",
    )
)

healthy_fit_qc

In [ ]:
from __future__ import annotations

from collections.abc import Mapping

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm, truncnorm


def plot_fit_parameter_histograms(
    parameters: Mapping[str, np.ndarray],
    *,
    bins: int = 80,
    width_factor: float = 2.0,
    fwhm_unit: str = "Hz",
    shift_unit: str = "Hz",
    title: str | None = (
        "LCModel fit parameters and proposed "
        "simulation distributions"
    ),
    display_quantiles: (
        tuple[float, float] | None
    ) = (0.001, 0.999),
) -> tuple[
    plt.Figure,
    np.ndarray,
]:
    """
    Plot FWHM and frequency-shift histograms.

    FWHM simulation:
        zero-truncated normal distribution

    Shift simulation:
        ordinary normal distribution

    For both parameters:
        location = in-vivo median
        scale = width_factor * in-vivo IQR
    """
    fwhm_values = np.asarray(
        parameters["FWHM"],
        dtype=np.float64,
    ).ravel()

    shift_values = np.asarray(
        parameters["shift"],
        dtype=np.float64,
    ).ravel()

    fwhm_values = fwhm_values[
        np.isfinite(fwhm_values)
        & (fwhm_values > 0)
    ]

    shift_values = shift_values[
        np.isfinite(shift_values)
    ]

    if fwhm_values.size == 0:
        raise ValueError(
            "No valid FWHM values."
        )

    if shift_values.size == 0:
        raise ValueError(
            "No valid shift values."
        )

    if width_factor <= 0:
        raise ValueError(
            "width_factor must be positive."
        )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 4.8),
        constrained_layout=True,
    )

    # ==============================================================
    # FWHM
    # ==============================================================
    fwhm_median = float(
        np.median(fwhm_values)
    )

    fwhm_q1, fwhm_q3 = np.quantile(
        fwhm_values,
        [0.25, 0.75],
    )

    fwhm_iqr = float(
        fwhm_q3 - fwhm_q1
    )

    fwhm_scale = (
        width_factor
        * fwhm_iqr
    )

    fwhm_lower_standardized = (
        -fwhm_median
        / fwhm_scale
    )

    fwhm_distribution = truncnorm(
        a=fwhm_lower_standardized,
        b=np.inf,
        loc=fwhm_median,
        scale=fwhm_scale,
    )

    if display_quantiles is None:
        fwhm_data_min = float(
            np.min(fwhm_values)
        )
        fwhm_data_max = float(
            np.max(fwhm_values)
        )
    else:
        fwhm_data_min, fwhm_data_max = (
            np.quantile(
                fwhm_values,
                display_quantiles,
            )
        )

    fwhm_x_min = 0.0

    fwhm_x_max = max(
        float(fwhm_data_max),
        float(
            fwhm_distribution.ppf(
                0.999
            )
        ),
    )

    axes[0].hist(
        fwhm_values,
        bins=bins,
        range=(
            fwhm_x_min,
            fwhm_x_max,
        ),
        density=True,
        alpha=0.65,
        label="In vivo",
    )

    fwhm_x = np.linspace(
        fwhm_x_min,
        fwhm_x_max,
        600,
    )

    axes[0].plot(
        fwhm_x,
        fwhm_distribution.pdf(
            fwhm_x
        ),
        linewidth=2,
        label=(
            "Simulation\n"
            f"location={fwhm_median:.3g}, "
            f"scale={fwhm_scale:.3g}"
        ),
    )

    axes[0].axvline(
        fwhm_median,
        linestyle="--",
        linewidth=1.3,
        label=(
            f"Median: "
            f"{fwhm_median:.3g}"
        ),
    )

    axes[0].set_title(
        "FWHM\n"
        f"n={fwhm_values.size:,}, "
        f"IQR={fwhm_iqr:.3g}"
    )

    axes[0].set_xlabel(
        f"FWHM [{fwhm_unit}]"
    )

    axes[0].set_ylabel(
        "Density"
    )

    axes[0].grid(
        axis="y",
        alpha=0.2,
    )

    axes[0].legend(
        frameon=False,
    )

    # ==============================================================
    # Frequency shift
    # ==============================================================
    shift_median = float(
        np.median(shift_values)
    )

    shift_q1, shift_q3 = np.quantile(
        shift_values,
        [0.25, 0.75],
    )

    shift_iqr = float(
        shift_q3 - shift_q1
    )

    shift_scale = (
        width_factor
        * shift_iqr
    )

    shift_distribution = norm(
        loc=shift_median,
        scale=shift_scale,
    )

    if display_quantiles is None:
        shift_data_min = float(
            np.min(shift_values)
        )
        shift_data_max = float(
            np.max(shift_values)
        )
    else:
        shift_data_min, shift_data_max = (
            np.quantile(
                shift_values,
                display_quantiles,
            )
        )

    shift_x_min = min(
        float(shift_data_min),
        float(
            shift_distribution.ppf(
                0.001
            )
        ),
    )

    shift_x_max = max(
        float(shift_data_max),
        float(
            shift_distribution.ppf(
                0.999
            )
        ),
    )

    axes[1].hist(
        shift_values,
        bins=bins,
        range=(
            shift_x_min,
            shift_x_max,
        ),
        density=True,
        alpha=0.65,
        label="In vivo",
    )

    shift_x = np.linspace(
        shift_x_min,
        shift_x_max,
        600,
    )

    axes[1].plot(
        shift_x,
        shift_distribution.pdf(
            shift_x
        ),
        linewidth=2,
        label=(
            "Simulation\n"
            f"location={shift_median:.3g}, "
            f"scale={shift_scale:.3g}"
        ),
    )

    axes[1].axvline(
        shift_median,
        linestyle="--",
        linewidth=1.3,
        label=(
            f"Median: "
            f"{shift_median:.3g}"
        ),
    )

    axes[1].set_title(
        "Frequency shift\n"
        f"n={shift_values.size:,}, "
        f"IQR={shift_iqr:.3g}"
    )

    axes[1].set_xlabel(
        f"Frequency shift [{shift_unit}]"
    )

    axes[1].set_ylabel(
        "Density"
    )

    axes[1].grid(
        axis="y",
        alpha=0.2,
    )

    axes[1].legend(
        frameon=False,
    )

    if title is not None:
        fig.suptitle(
            title,
            fontsize=15,
        )

    return fig, axes

In [ ]:
fig, axes = plot_fit_parameter_histograms(
    healthy_fit_parameters,
    width_factor=2.0,
    fwhm_unit="Hz",
    shift_unit="Hz",
    title=(
        "Healthy LCModel parameters – "
        "simulation scale = 2 × IQR"
    ),
)

plt.show()

In [ ]:
fwhm = healthy_fit_parameters["FWHM"]
fwhm = fwhm[np.isfinite(fwhm)]

unique_values, counts = np.unique(fwhm, return_counts=True)

print("Number of voxels:", fwhm.size)
print("Number of unique values:", unique_values.size)
print("First unique values:")
print(unique_values[:30])

In [ ]:
differences = np.diff(unique_values)
positive_differences = differences[differences > 0]

print("Smallest spacings:")
print(np.sort(positive_differences)[:30])

In [ ]:
from __future__ import annotations

from pathlib import Path

import h5py
import numpy as np


def estimate_water_lipid_ratios_from_walinet(
    data_path: str | Path,
    data_after_walinet_path: str | Path,
    resource_path: str | Path,
    *,
    denominator_epsilon: float = 1e-12,
    n_example_voxels: int = 8,
) -> tuple[
    dict[str, np.ndarray],
    dict[str, np.ndarray],
    dict[str, object],
]:
    """
    Estimate water/metabolite and lipid/metabolite amplitude ratios.

    Expected decomposition
    ----------------------
    total FID:
        data.npy

    metabolite FID:
        data_after_walinet.npy

    water FID:
        water_fids from the simulation-resource HDF5

    lipid spectrum:
        total spectrum - water spectrum - metabolite spectrum

    Ratios
    ------
        max(abs(water spectrum))
        ------------------------
        max(abs(metabolite spectrum))

    and

        max(abs(lipid spectrum))
        ------------------------
        max(abs(metabolite spectrum))

    Notes
    -----
    All three FID sources must have identical scaling, shape and spectral
    ordering.
    """
    data_path = Path(data_path)
    data_after_walinet_path = Path(data_after_walinet_path)
    resource_path = Path(resource_path)

    total_fids = np.load(
        data_path,
        mmap_mode="r",
    )

    metabolite_fids = np.load(
        data_after_walinet_path,
        mmap_mode="r",
    )

    if total_fids.shape != metabolite_fids.shape:
        raise ValueError(
            "Shape mismatch between total data and WALINET output:\n"
            f"  total:   {total_fids.shape}\n"
            f"  WALINET: {metabolite_fids.shape}"
        )

    if total_fids.ndim != 4:
        raise ValueError(
            "Expected data with shape (x, y, z, spectral_points), "
            f"found {total_fids.shape}."
        )

    if not np.iscomplexobj(total_fids):
        raise TypeError(
            "data.npy is expected to contain complex FIDs."
        )

    if not np.iscomplexobj(metabolite_fids):
        raise TypeError(
            "data_after_walinet.npy is expected to contain complex FIDs."
        )

    spatial_shape = total_fids.shape[:-1]
    n_spectral_points = total_fids.shape[-1]

    water_ratio_parts: list[np.ndarray] = []
    lipid_ratio_parts: list[np.ndarray] = []

    example_total: list[np.ndarray] = []
    example_water: list[np.ndarray] = []
    example_metabolites: list[np.ndarray] = []
    example_lipids: list[np.ndarray] = []

    n_mask_voxels = 0
    n_valid_voxels = 0

    with h5py.File(resource_path, "r") as h5_file:
        brain_mask = np.asarray(
            h5_file["brain_mask"],
            dtype=bool,
        )

        water_dataset = h5_file["water_fids"]

        if brain_mask.shape != spatial_shape:
            raise ValueError(
                "Brain-mask shape mismatch:\n"
                f"  data: {spatial_shape}\n"
                f"  mask: {brain_mask.shape}"
            )

        if water_dataset.shape != total_fids.shape:
            raise ValueError(
                "Water-FID shape mismatch:\n"
                f"  total: {total_fids.shape}\n"
                f"  water: {water_dataset.shape}"
            )

        # Process one spatial z-slice at a time.
        for z_index in range(spatial_shape[2]):
            mask_slice = brain_mask[:, :, z_index]

            if not np.any(mask_slice):
                continue

            total_slice = np.asarray(
                total_fids[:, :, z_index, :]
            )[mask_slice]

            metabolite_slice = np.asarray(
                metabolite_fids[:, :, z_index, :]
            )[mask_slice]

            water_slice = np.asarray(
                water_dataset[:, :, z_index, :]
            )[mask_slice]

            n_mask_voxels += total_slice.shape[0]

            # FID -> frequency domain.
            # fftshift is unnecessary for max(abs(...)).
            total_spectra = np.fft.fft(
                total_slice,
                axis=-1,
            )

            water_spectra = np.fft.fft(
                water_slice,
                axis=-1,
            )

            metabolite_spectra = np.fft.fft(
                metabolite_slice,
                axis=-1,
            )

            lipid_spectra = (
                total_spectra
                - water_spectra
                - metabolite_spectra
            )

            water_max_abs = np.max(
                np.abs(water_spectra),
                axis=-1,
            )

            lipid_max_abs = np.max(
                np.abs(lipid_spectra),
                axis=-1,
            )

            metabolite_max_abs = np.max(
                np.abs(metabolite_spectra),
                axis=-1,
            )

            valid = (
                np.isfinite(water_max_abs)
                & np.isfinite(lipid_max_abs)
                & np.isfinite(metabolite_max_abs)
                & (metabolite_max_abs > denominator_epsilon)
            )

            water_ratios = (
                water_max_abs[valid]
                / metabolite_max_abs[valid]
            )

            lipid_ratios = (
                lipid_max_abs[valid]
                / metabolite_max_abs[valid]
            )

            water_ratio_parts.append(
                water_ratios.astype(
                    np.float64,
                    copy=False,
                )
            )

            lipid_ratio_parts.append(
                lipid_ratios.astype(
                    np.float64,
                    copy=False,
                )
            )

            n_valid_voxels += int(valid.sum())

            # Store a few spectra for visual inspection.
            n_missing_examples = (
                n_example_voxels
                - len(example_total)
            )

            if n_missing_examples > 0:
                valid_indices = np.flatnonzero(valid)[
                    :n_missing_examples
                ]

                example_total.extend(
                    total_spectra[valid_indices]
                )
                example_water.extend(
                    water_spectra[valid_indices]
                )
                example_metabolites.extend(
                    metabolite_spectra[valid_indices]
                )
                example_lipids.extend(
                    lipid_spectra[valid_indices]
                )

    ratios = {
        "water_to_metabolite": np.concatenate(
            water_ratio_parts
        ),
        "lipid_to_metabolite": np.concatenate(
            lipid_ratio_parts
        ),
    }

    example_spectra = {
        "total": np.asarray(example_total),
        "water": np.asarray(example_water),
        "metabolites": np.asarray(example_metabolites),
        "lipids": np.asarray(example_lipids),
    }

    diagnostics = {
        "data_shape": tuple(total_fids.shape),
        "n_spectral_points": n_spectral_points,
        "n_mask_voxels": n_mask_voxels,
        "n_valid_voxels": n_valid_voxels,
        "n_example_voxels": len(example_total),
    }

    return ratios, example_spectra, diagnostics

In [ ]:
from pathlib import Path


BASE_PATH = Path(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/"
    "mrsbrain/public/hfish/Denoising/datasets/Proton/7T/"
    "NoB0Correction/Vol1_Brisbane"
)

DATA_PATH = (
    BASE_PATH
    / "OriginalData"
    / "data.npy"
)

DATA_AFTER_WALINET_PATH = (
    BASE_PATH
    / "OriginalData"
    / "data_after_walinet.npy"
)

RESOURCE_PATH = (
    BASE_PATH
    / "TrainData"
    / "SimulationResources_water_lipid_v1.h5"
)


ratios, example_spectra, diagnostics = (
    estimate_water_lipid_ratios_from_walinet(
        DATA_PATH,
        DATA_AFTER_WALINET_PATH,
        RESOURCE_PATH,
        n_example_voxels=8,
    )
)

diagnostics

In [ ]:
for name, values in ratios.items():
    q1, median, q3 = np.quantile(
        values,
        [0.25, 0.50, 0.75],
    )

    print(name)
    print(f"  n:      {values.size:,}")
    print(f"  median: {median:.6g}")
    print(f"  IQR:    {q3 - q1:.6g}")
    print(f"  min:    {values.min():.6g}")
    print(f"  max:    {values.max():.6g}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_walinet_decomposition(
    spectra: dict[str, np.ndarray],
) -> tuple[plt.Figure, np.ndarray]:
    n_voxels = spectra["total"].shape[0]

    fig, axes = plt.subplots(
        n_voxels,
        1,
        figsize=(12, 3 * n_voxels),
        squeeze=False,
        constrained_layout=True,
    )

    for voxel_index, axis in enumerate(axes[:, 0]):
        # Shift only for display.
        total = np.fft.fftshift(
            spectra["total"][voxel_index]
        )
        water = np.fft.fftshift(
            spectra["water"][voxel_index]
        )
        metabolites = np.fft.fftshift(
            spectra["metabolites"][voxel_index]
        )
        lipids = np.fft.fftshift(
            spectra["lipids"][voxel_index]
        )

        axis.plot(
            np.abs(total),
            label="Total",
        )
        axis.plot(
            np.abs(water),
            label="Hankel water",
        )
        axis.plot(
            np.abs(metabolites),
            label="After WALINET",
        )
        axis.plot(
            np.abs(lipids),
            label="Residual lipids",
        )

        axis.set_title(
            f"Example voxel {voxel_index}"
        )
        axis.set_xlabel("Spectral sample")
        axis.set_ylabel("Absolute amplitude")
        axis.grid(axis="y", alpha=0.2)
        axis.legend(frameon=False)

    return fig, axes

In [ ]:
fig, axes = plot_walinet_decomposition(
    example_spectra
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import truncnorm


def plot_water_lipid_ratio_distributions(
    ratios: dict[str, np.ndarray],
    *,
    bins: int = 80,
    width_factor: float = 2.0,
    display_quantiles: tuple[float, float] = (
        0.001,
        0.999,
    ),
) -> tuple[plt.Figure, np.ndarray]:
    definitions = [
        (
            "water_to_metabolite",
            "Water / metabolites",
            r"$\max|S_{\mathrm{water}}|"
            r"/\max|S_{\mathrm{metab}}|$",
        ),
        (
            "lipid_to_metabolite",
            "Lipids / metabolites",
            r"$\max|S_{\mathrm{lipid}}|"
            r"/\max|S_{\mathrm{metab}}|$",
        ),
    ]

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 5),
        constrained_layout=True,
    )

    for axis, (
        key,
        panel_title,
        x_label,
    ) in zip(axes, definitions):
        values = np.asarray(
            ratios[key],
            dtype=np.float64,
        )

        values = values[
            np.isfinite(values)
            & (values >= 0)
        ]

        q1, median, q3 = np.quantile(
            values,
            [0.25, 0.50, 0.75],
        )

        iqr = float(q3 - q1)

        simulation_scale = (
            width_factor * iqr
        )

        if simulation_scale <= 0:
            raise ValueError(
                f"IQR is zero for {key}."
            )

        lower_bound_standardized = (
            -median
            / simulation_scale
        )

        distribution = truncnorm(
            a=lower_bound_standardized,
            b=np.inf,
            loc=median,
            scale=simulation_scale,
        )

        _, data_max = np.quantile(
            values,
            display_quantiles,
        )

        x_max = max(
            float(data_max),
            float(distribution.ppf(0.999)),
        )

        axis.hist(
            values,
            bins=bins,
            range=(0.0, x_max),
            density=True,
            alpha=0.65,
            label="In-vivo estimate",
        )

        x = np.linspace(
            0.0,
            x_max,
            600,
        )

        axis.plot(
            x,
            distribution.pdf(x),
            linewidth=2.2,
            label=(
                "Simulation\n"
                f"location={median:.3g}, "
                f"scale={simulation_scale:.3g}"
            ),
        )

        axis.axvline(
            median,
            linestyle="--",
            linewidth=1.3,
            label=f"Median: {median:.3g}",
        )

        axis.axvline(
            q1,
            linestyle=":",
            linewidth=1.0,
        )

        axis.axvline(
            q3,
            linestyle=":",
            linewidth=1.0,
            label=f"IQR: {iqr:.3g}",
        )

        axis.set_xlim(0.0, x_max)
        axis.set_xlabel(x_label)
        axis.set_ylabel("Density")

        axis.set_title(
            f"{panel_title}\n"
            f"n={values.size:,}, "
            f"IQR={iqr:.3g}"
        )

        axis.grid(
            axis="y",
            alpha=0.2,
        )

        axis.legend(
            frameon=False,
        )

    fig.suptitle(
        "WALINET-based nuisance-signal scaling – "
        f"simulation scale = {width_factor:g} × IQR",
        fontsize=15,
    )

    return fig, axes

In [ ]:
fig, axes = plot_water_lipid_ratio_distributions(
    ratios,
    bins=80,
    width_factor=2.0,
)

plt.show()